# 🌸 Iris Flower Classification
### CodeAlpha Data Science Internship — Task 1
**Author:** Sanjai KV | **GitHub:** sanjaikv2255

---
**Objective:** Train and evaluate machine learning models to classify Iris flowers into three species — *Setosa*, *Versicolor*, and *Virginica* — based on sepal and petal measurements.

**Models Used:** K-Nearest Neighbors · Decision Tree · Support Vector Machine

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f9f9f9',
    'axes.grid': True, 'grid.color': '#e0e0e0',
    'axes.spines.top': False, 'axes.spines.right': False
})
COLORS = {'Iris-setosa': '#4C72B0', 'Iris-versicolor': '#DD8452', 'Iris-virginica': '#55A868'}
print('✅ Libraries imported successfully')

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('Iris.csv')
df.drop('Id', axis=1, inplace=True)

print('Shape:', df.shape)
print('\nClass Distribution:')
print(df['Species'].value_counts())
print('\nMissing Values:', df.isnull().sum().sum())
df.head()

In [ ]:
df.describe().round(2)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Pairplot
features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
labels = df['Species'].unique()
fig, axes = plt.subplots(4, 4, figsize=(13, 12))

for i, fx in enumerate(features):
    for j, fy in enumerate(features):
        ax = axes[i][j]
        if i == j:
            for sp in labels:
                ax.hist(df[df['Species']==sp][fx], bins=15, alpha=0.65, color=COLORS[sp], edgecolor='white')
        else:
            for sp in labels:
                sub = df[df['Species']==sp]
                ax.scatter(sub[fy], sub[fx], alpha=0.7, s=25, color=COLORS[sp], edgecolors='white', linewidths=0.4)
        if i == 3: ax.set_xlabel(fy.replace('Cm',''), fontsize=8)
        if j == 0: ax.set_ylabel(fx.replace('Cm',''), fontsize=8)
        ax.tick_params(labelsize=7)

patches = [mpatches.Patch(color=COLORS[s], label=s.replace('Iris-','')) for s in labels]
fig.legend(handles=patches, loc='upper center', ncol=3, fontsize=10, bbox_to_anchor=(0.5,1.01), frameon=False)
fig.suptitle('Pairplot — Iris Features by Species', fontsize=14, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig('plots/01_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation Heatmap
fig, ax = plt.subplots(figsize=(7, 5))
corr = df.drop('Species', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask, ax=ax,
            linewidths=0.5, cbar_kws={'shrink':0.8}, annot_kws={'size':11})
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('plots/02_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n🔍 Key Insight: Petal Length & Petal Width are highly correlated (r=0.96)')
print('   Petal features are the most discriminative for species classification.')

In [ ]:
# Boxplots
fig, axes = plt.subplots(1, 4, figsize=(14, 5))
for ax, feat in zip(axes, features):
    data = [df[df['Species']==sp][feat].values for sp in labels]
    bp = ax.boxplot(data, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2))
    for patch, sp in zip(bp['boxes'], labels):
        patch.set_facecolor(COLORS[sp]); patch.set_alpha(0.75)
    ax.set_xticklabels([s.replace('Iris-','') for s in labels], fontsize=8.5)
    ax.set_title(feat.replace('Cm',' (cm)'), fontsize=10, fontweight='bold')
fig.suptitle('Feature Distribution by Species', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/03_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preprocessing

In [ ]:
X = df.drop('Species', axis=1)
le = LabelEncoder()
y = le.fit_transform(df['Species'])  # setosa=0, versicolor=1, virginica=2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train size: {len(X_train)} samples')
print(f'Test size : {len(X_test)} samples')
print(f'Classes   : {list(le.classes_)}')

## 5. Model Training & Evaluation

In [ ]:
models = {
    'K-Nearest Neighbors':      KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':             DecisionTreeClassifier(max_depth=4, random_state=42),
    'Support Vector Machine':    SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred) * 100
    results[name] = {'model': model, 'acc': acc, 'y_pred': y_pred}
    print(f'{'─'*45}')
    print(f'  {name}  →  Accuracy: {acc:.2f}%')
    print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

In [ ]:
# Accuracy Comparison Bar Chart
fig, ax = plt.subplots(figsize=(8, 5))
names = list(results.keys())
accs  = [results[n]['acc'] for n in names]
bars = ax.barh(names, accs, color=['#4C72B0','#DD8452','#55A868'], height=0.5, edgecolor='white')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_width()-1.5, bar.get_y()+bar.get_height()/2,
            f'{acc:.1f}%', va='center', ha='right', fontsize=11, color='white', fontweight='bold')
ax.set_xlim(85, 102)
ax.set_xlabel('Accuracy (%)', fontsize=11)
ax.set_title('Model Accuracy Comparison', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('plots/04_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
class_names = [c.replace('Iris-','') for c in le.classes_]

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, cbar=False, annot_kws={'size':13})
    ax.set_title(f'{name}\n({res["acc"]:.1f}%)', fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=9); ax.set_ylabel('Actual', fontsize=9)

fig.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/05_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Decision Boundary (Petal Length vs Petal Width — best 2 features)
best_model_name = max(results, key=lambda k: results[k]['acc'])
best_model = results[best_model_name]['model']

X2 = df[['PetalLengthCm','PetalWidthCm']].values
X2_train, X2_test, y2_train, _ = train_test_split(X2, y, test_size=0.2, random_state=42, stratify=y)
best_2f = type(best_model)(**{k:v for k,v in best_model.get_params().items()})
best_2f.fit(X2_train, y2_train)

h = 0.02
x_min, x_max = X2[:,0].min()-0.5, X2[:,0].max()+0.5
y_min, y_max = X2[:,1].min()-0.5, X2[:,1].max()+0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = best_2f.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.35, cmap=plt.cm.get_cmap('Pastel1', 3))
for sp in labels:
    mask = df['Species']==sp
    ax.scatter(df[mask]['PetalLengthCm'], df[mask]['PetalWidthCm'],
               label=sp.replace('Iris-',''), color=COLORS[sp], edgecolors='white', linewidths=0.5, s=55, alpha=0.9)
ax.set_xlabel('Petal Length (cm)', fontsize=11)
ax.set_ylabel('Petal Width (cm)', fontsize=11)
ax.set_title(f'Decision Boundary — {best_model_name}', fontsize=12, fontweight='bold')
ax.legend(title='Species', fontsize=9)
plt.tight_layout()
plt.savefig('plots/06_decision_boundary.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Conclusions

In [ ]:
print('=' * 55)
print('  FINAL RESULTS SUMMARY')
print('=' * 55)
for name, res in results.items():
    print(f'  {name:<30} → {res["acc"]:.2f}%')
best = max(results, key=lambda k: results[k]['acc'])
print(f'\n🏆 Best Model: {best} ({results[best]["acc"]:.2f}%)')
print('''
Key Findings:
  • Iris-setosa is perfectly separable from the other two species
  • Petal Length and Petal Width are the most informative features
  • KNN achieved 100% accuracy on this dataset with k=5
  • All 3 models performed strongly (93%+), confirming the dataset is well-structured
''')